## Cell 1 — Imports and Dependencies

In [38]:
!pip install web3 python-dotenv

In [39]:
import os
import json
import hashlib
from web3 import Web3
from dotenv import load_dotenv
from datetime import datetime

# Load environment variables from .env file
load_dotenv()

print("Libraries loaded successfully.")


Libraries loaded successfully.


In [40]:
import os
print(os.getcwd())

C:\Users\enios


## Cell 2 — Configuration and Connection

In [41]:
# ── RPC and Contract Configuration ──
RPC_URL          = os.getenv("RPC_URL")  # set this in your .env file
CONTRACT_ADDRESS = "0x728F1cDc7934f7aD24A75425DC48b414F207d8d6"

# ── Load private key from .env file (place it in the project root) ──
load_dotenv()
PRIVATE_KEY = os.getenv("PRIVATE_KEY")

if not PRIVATE_KEY:
    raise ValueError("PRIVATE_KEY not found in .env file. Please create a .env file.")

# ── Connect to Sepolia via Alchemy ──
w3 = Web3(Web3.HTTPProvider(RPC_URL))

if w3.is_connected():
    print(f"Connected to Sepolia network")
    print(f"Chain ID       : {w3.eth.chain_id}")
    print(f"Latest block   : {w3.eth.block_number}")
else:
    raise ConnectionError("Failed to connect to Sepolia. Check your RPC URL.")

# ── Derive wallet address from private key ──
account        = w3.eth.account.from_key(PRIVATE_KEY)
WALLET_ADDRESS = account.address
print(f"Wallet address : {WALLET_ADDRESS}")
print(f"ETH balance    : {w3.from_wei(w3.eth.get_balance(WALLET_ADDRESS), 'ether'):.4f} ETH")

Connected to Sepolia network
Chain ID       : 11155111
Latest block   : 10973826
Wallet address : 0x84f4E63439ea5F48cB137a72c8e98DFa68Cb6A8a
ETH balance    : 0.4987 ETH


## Cell 3 — Load ABI and Create Contract Instance

In [42]:
# Full ABI for PatientRecordContract
CONTRACT_ABI = json.loads('''
[
  {"inputs":[{"internalType":"string","name":"_patientId","type":"string"},{"internalType":"string","name":"_providerName","type":"string"},{"internalType":"string","name":"_hospital","type":"string"},{"internalType":"string","name":"_diagnosisCode","type":"string"},{"internalType":"bytes32","name":"_recordHash","type":"bytes32"}],"name":"addRecord","outputs":[{"internalType":"uint256","name":"newRecordId","type":"uint256"}],"stateMutability":"nonpayable","type":"function"},
  {"inputs":[{"internalType":"address","name":"_provider","type":"address"}],"name":"authoriseProvider","outputs":[],"stateMutability":"nonpayable","type":"function"},
  {"inputs":[],"stateMutability":"nonpayable","type":"constructor"},
  {"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"provider","type":"address"},{"indexed":false,"internalType":"uint256","name":"timestamp","type":"uint256"}],"name":"ProviderAuthorised","type":"event"},
  {"anonymous":false,"inputs":[{"indexed":true,"internalType":"address","name":"provider","type":"address"},{"indexed":false,"internalType":"uint256","name":"timestamp","type":"uint256"}],"name":"ProviderRevoked","type":"event"},
  {"anonymous":false,"inputs":[{"indexed":true,"internalType":"uint256","name":"recordId","type":"uint256"},{"indexed":true,"internalType":"string","name":"patientId","type":"string"},{"indexed":true,"internalType":"address","name":"provider","type":"address"},{"indexed":false,"internalType":"uint256","name":"timestamp","type":"uint256"}],"name":"RecordAdded","type":"event"},
  {"anonymous":false,"inputs":[{"indexed":true,"internalType":"uint256","name":"recordId","type":"uint256"},{"indexed":true,"internalType":"address","name":"fromProvider","type":"address"},{"indexed":true,"internalType":"address","name":"toProvider","type":"address"},{"indexed":false,"internalType":"uint256","name":"timestamp","type":"uint256"}],"name":"RecordTransferred","type":"event"},
  {"inputs":[{"internalType":"address","name":"_provider","type":"address"}],"name":"revokeProvider","outputs":[],"stateMutability":"nonpayable","type":"function"},
  {"inputs":[{"internalType":"uint256","name":"_recordId","type":"uint256"},{"internalType":"address","name":"_toProvider","type":"address"}],"name":"transferRecord","outputs":[],"stateMutability":"nonpayable","type":"function"},
  {"inputs":[],"name":"admin","outputs":[{"internalType":"address","name":"","type":"address"}],"stateMutability":"view","type":"function"},
  {"inputs":[{"internalType":"address","name":"","type":"address"}],"name":"authorisedProviders","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"view","type":"function"},
  {"inputs":[{"internalType":"string","name":"_patientId","type":"string"}],"name":"getPatientHistory","outputs":[{"internalType":"uint256[]","name":"","type":"uint256[]"}],"stateMutability":"view","type":"function"},
  {"inputs":[{"internalType":"uint256","name":"_recordId","type":"uint256"}],"name":"getRecord","outputs":[{"components":[{"internalType":"uint256","name":"recordId","type":"uint256"},{"internalType":"string","name":"patientId","type":"string"},{"internalType":"string","name":"providerName","type":"string"},{"internalType":"string","name":"hospital","type":"string"},{"internalType":"string","name":"diagnosisCode","type":"string"},{"internalType":"bytes32","name":"recordHash","type":"bytes32"},{"internalType":"address","name":"currentProvider","type":"address"},{"internalType":"uint256","name":"timestamp","type":"uint256"},{"internalType":"bool","name":"isActive","type":"bool"}],"internalType":"struct PatientRecordContract.PatientRecord","name":"","type":"tuple"}],"stateMutability":"view","type":"function"},
  {"inputs":[{"internalType":"address","name":"_provider","type":"address"}],"name":"isAuthorised","outputs":[{"internalType":"bool","name":"","type":"bool"}],"stateMutability":"view","type":"function"},
  {"inputs":[{"internalType":"uint256","name":"","type":"uint256"}],"name":"patientRecords","outputs":[{"internalType":"uint256","name":"recordId","type":"uint256"},{"internalType":"string","name":"patientId","type":"string"},{"internalType":"string","name":"providerName","type":"string"},{"internalType":"string","name":"hospital","type":"string"},{"internalType":"string","name":"diagnosisCode","type":"string"},{"internalType":"bytes32","name":"recordHash","type":"bytes32"},{"internalType":"address","name":"currentProvider","type":"address"},{"internalType":"uint256","name":"timestamp","type":"uint256"},{"internalType":"bool","name":"isActive","type":"bool"}],"stateMutability":"view","type":"function"},
  {"inputs":[],"name":"totalRecords","outputs":[{"internalType":"uint256","name":"","type":"uint256"}],"stateMutability":"view","type":"function"}
]
''')

# ── Create contract instance using address and ABI ──
contract = w3.eth.contract(
    address=Web3.to_checksum_address(CONTRACT_ADDRESS),
    abi=CONTRACT_ABI
)

# ── Verify contract is reachable by reading admin address ──
admin_address = contract.functions.admin().call()
total_records  = contract.functions.totalRecords().call()

print(f"Contract loaded successfully")
print(f"Contract address : {CONTRACT_ADDRESS}")
print(f"Admin address    : {admin_address}")
print(f"Total records    : {total_records}")


Contract loaded successfully
Contract address : 0x728F1cDc7934f7aD24A75425DC48b414F207d8d6
Admin address    : 0x84f4E63439ea5F48cB137a72c8e98DFa68Cb6A8a
Total records    : 7


## Cell 4 — Transaction Helper Functions

In [43]:
import time

def send_transaction(tx_function):
    """
    Builds, signs, and sends a transaction to the Sepolia network.
    Fetches a fresh nonce before every transaction to prevent nonce conflicts.
    """
    # Always fetch the latest nonce fresh from the network
    nonce = w3.eth.get_transaction_count(WALLET_ADDRESS, 'pending')

    estimated_gas = tx_function.estimate_gas({"from": WALLET_ADDRESS})
    gas_limit     = int(estimated_gas * 1.2)

    tx = tx_function.build_transaction({
        "from"    : WALLET_ADDRESS,
        "nonce"   : nonce,
        "gas"     : gas_limit,
        "chainId" : 11155111
    })

    signed_tx = w3.eth.account.sign_transaction(tx, PRIVATE_KEY)
    tx_hash   = w3.eth.send_raw_transaction(signed_tx.raw_transaction)
    print(f"  Transaction sent   : {tx_hash.hex()}")
    print(f"  Waiting for mining ...")

    receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=120)
    status  = "SUCCESS" if receipt.status == 1 else "FAILED"
    print(f"  Status             : {status}")
    print(f"  Block number       : {receipt.blockNumber}")
    print(f"  Gas used           : {receipt.gasUsed}")

    # Brief pause to allow the network to update the nonce
    time.sleep(2)

    return receipt

def generate_record_hash(patient_id, provider_name, diagnosis_code, timestamp):
    """
    Generates a SHA-256 hash representing a patient record.
    In production this would hash the full clinical document.
    Returns a bytes32-compatible value for the smart contract.
    """
    raw = f"{patient_id}:{provider_name}:{diagnosis_code}:{timestamp}"
    return hashlib.sha256(raw.encode()).digest()

print("Helper functions updated successfully.")

print("Helper functions updated successfully.")

Helper functions updated successfully.
Helper functions updated successfully.


## Cell 5 — Authorise a Second Healthcare Provider

In [44]:
# ── Second provider address (a separate Ethereum account) ──
# This simulates a second healthcare provider on the network.
# Replace with a real second MetaMask account address if available,
# or use this placeholder address for demonstration purposes.
SECOND_PROVIDER = Web3.to_checksum_address("0x742d35cc6634c0532925a3b8d4c9b8e8b6b0e1a2")

print("Authorising second provider...")
print(f"  Provider address : {SECOND_PROVIDER}")

# Check if already authorised before sending a transaction
already_authorised = contract.functions.isAuthorised(SECOND_PROVIDER).call()

if already_authorised:
    print(f"  Status           : Already authorised — skipping transaction")
else:
    # Build the authoriseProvider transaction
    tx_function = contract.functions.authoriseProvider(SECOND_PROVIDER)
    receipt     = send_transaction(tx_function)
    print(f"  Transaction hash : {receipt.transactionHash.hex()}")

# Confirm authorisation status
is_authorised = contract.functions.isAuthorised(SECOND_PROVIDER).call()
print(f"  Authorised       : {is_authorised}")


Authorising second provider...
  Provider address : 0x742d35CC6634c0532925a3B8D4c9b8E8B6b0e1A2
  Status           : Already authorised — skipping transaction
  Authorised       : True


## Cell 6 — Add Patient Records to the Contract

In [45]:
# ── Patient records to add ──
records_to_add = [
    {
        "patient_id"    : "P001",
        "provider_name" : "Dr. Sarah Smith",
        "hospital"      : "City General Hospital",
        "diagnosis_code": "J18.9"
    },
    {
        "patient_id"    : "P002",
        "provider_name" : "Dr. James Patel",
        "hospital"      : "North Clinic",
        "diagnosis_code": "E11.9"
    },
    {
        "patient_id"    : "P003",
        "provider_name" : "Dr. Sarah Smith",
        "hospital"      : "City General Hospital",
        "diagnosis_code": "I10"
    }
]

added_record_ids = []
total_to_add = len(records_to_add)
print(f"Adding {total_to_add} patient records to the blockchain...")
print("")

for i, record in enumerate(records_to_add):
    pid    = record["patient_id"]
    pname  = record["provider_name"]
    pcode  = record["diagnosis_code"]
    print(f"[Record {i+1}] Patient: {pid} | Provider: {pname} | Diagnosis: {pcode}")

    timestamp   = int(datetime.now().timestamp())
    record_hash = generate_record_hash(pid, pname, pcode, timestamp)

    tx_function = contract.functions.addRecord(
        record["patient_id"],
        record["provider_name"],
        record["hospital"],
        record["diagnosis_code"],
        record_hash
    )

    receipt = send_transaction(tx_function)
    txhash  = receipt.transactionHash.hex()
    print(f"  Transaction hash : {txhash}")
    print("")

total = contract.functions.totalRecords().call()
print(f"Total records on contract : {total}")

Adding 3 patient records to the blockchain...

[Record 1] Patient: P001 | Provider: Dr. Sarah Smith | Diagnosis: J18.9
  Transaction sent   : 611601a580f89327dc37948955824a610cb317313a335e649fa47c85bbec13cf
  Waiting for mining ...
  Status             : SUCCESS
  Block number       : 10973828
  Gas used           : 266964
  Transaction hash : 611601a580f89327dc37948955824a610cb317313a335e649fa47c85bbec13cf

[Record 2] Patient: P002 | Provider: Dr. James Patel | Diagnosis: E11.9
  Transaction sent   : 3e8154ccc0750a038b3562a93d953dbd1fd050d0780e4e42938680e96e4a4baa
  Waiting for mining ...
  Status             : SUCCESS
  Block number       : 10973829
  Gas used           : 266856
  Transaction hash : 3e8154ccc0750a038b3562a93d953dbd1fd050d0780e4e42938680e96e4a4baa

[Record 3] Patient: P003 | Provider: Dr. Sarah Smith | Diagnosis: I10
  Transaction sent   : c0a43cd3cc63ccb75a1c38f4cc88a8b29f2a5037aa529d034c1160d4fd70a462
  Waiting for mining ...
  Status             : SUCCESS
  Block n

## Cell 7 — Retrieve and Display Records

In [46]:
total = contract.functions.totalRecords().call()
num_records = str(total)
print("Retrieving " + num_records + " records from the contract...")
print("=" * 65)

for record_id in range(1, total + 1):
    record = contract.functions.getRecord(record_id).call()

    print("Record ID       : " + str(record[0]))
    print("Patient ID      : " + str(record[1]))
    print("Provider Name   : " + str(record[2]))
    print("Hospital        : " + str(record[3]))
    print("Diagnosis Code  : " + str(record[4]))
    print("Record Hash     : 0x" + record[5].hex())
    print("Current Provider: " + str(record[6]))
    print("Timestamp       : " + datetime.fromtimestamp(record[7]).strftime('%Y-%m-%d %H:%M:%S'))
    print("Active          : " + str(record[8]))
    print("=" * 65)

Retrieving 10 records from the contract...
Record ID       : 1
Patient ID      : P001
Provider Name   : Dr. Sarah Smith
Hospital        : City General Hospital
Diagnosis Code  : J18.9
Record Hash     : 0xd3966ebc3a22609e34a37ac9ab8ad9f7d855067ed67ccf7810796addb1c372a5
Current Provider: 0x742d35CC6634c0532925a3B8D4c9b8E8B6b0e1A2
Timestamp       : 2026-05-30 13:48:00
Active          : True
Record ID       : 2
Patient ID      : P001
Provider Name   : Dr. Sarah Smith
Hospital        : City General Hospital
Diagnosis Code  : J18.9
Record Hash     : 0xccb559940d732899e34349ef387eb7d475f638e15d81b4248e4bd5974f0ec456
Current Provider: 0x84f4E63439ea5F48cB137a72c8e98DFa68Cb6A8a
Timestamp       : 2026-05-30 13:51:36
Active          : True
Record ID       : 3
Patient ID      : P002
Provider Name   : Dr. James Patel
Hospital        : North Clinic
Diagnosis Code  : E11.9
Record Hash     : 0x414b10e995c6236f4e4e56e9114f14cba3f0ab927129f96a076d8bfb68eacfd8
Current Provider: 0x84f4E63439ea5F48cB137a72

## Cell 8 — Transfer a Patient Record Between Providers

In [47]:
RECORD_ID_TO_TRANSFER = 1

print("Transferring Record " + str(RECORD_ID_TO_TRANSFER) + " to second provider...")
print("")

# Confirm current holder before transfer
record_before = contract.functions.getRecord(RECORD_ID_TO_TRANSFER).call()
print("  Before transfer:")
print("    Patient ID       : " + str(record_before[1]))
print("    Current Provider : " + str(record_before[6]))

# Build and send the transferRecord transaction
tx_function = contract.functions.transferRecord(
    RECORD_ID_TO_TRANSFER,
    SECOND_PROVIDER
)
receipt = send_transaction(tx_function)
print("  Transaction hash : " + receipt.transactionHash.hex())

# Confirm new holder after transfer
record_after = contract.functions.getRecord(RECORD_ID_TO_TRANSFER).call()
print("")
print("  After transfer:")
print("    Patient ID       : " + str(record_after[1]))
print("    Current Provider : " + str(record_after[6]))
print("")
print("  Transfer successful: " + str(record_before[6] != record_after[6]))

Transferring Record 1 to second provider...

  Before transfer:
    Patient ID       : P001
    Current Provider : 0x742d35CC6634c0532925a3B8D4c9b8E8B6b0e1A2


ContractLogicError: ('execution reverted: PatientRecordContract: only the current record holder can transfer', '0x08c379a00000000000000000000000000000000000000000000000000000000000000020000000000000000000000000000000000000000000000000000000000000004250617469656e745265636f7264436f6e74726163743a206f6e6c79207468652063757272656e74207265636f726420686f6c6465722063616e207472616e73666572000000000000000000000000000000000000000000000000000000000000')

## Cell 9 — Query Patient Record History

In [48]:
patients_to_check = ["P001", "P002", "P003"]

print("Querying patient record history...")
print("")

for patient_id in patients_to_check:
    record_ids  = contract.functions.getPatientHistory(patient_id).call()
    num_records = str(len(record_ids))
    print("Patient " + patient_id + " : " + num_records + " record(s) found — IDs: " + str(record_ids))

print("")
total_records = contract.functions.totalRecords().call()
admin_address = contract.functions.admin().call()
print("Total records on contract : " + str(total_records))
print("Admin address             : " + str(admin_address))

Querying patient record history...

Patient P001 : 4 record(s) found — IDs: [1, 2, 5, 8]
Patient P002 : 3 record(s) found — IDs: [3, 6, 9]
Patient P003 : 3 record(s) found — IDs: [4, 7, 10]

Total records on contract : 10
Admin address             : 0x84f4E63439ea5F48cB137a72c8e98DFa68Cb6A8a
